# X3 — Ampliando o golden dataset com dados sintéticos

**Pergunta:** como ter mais casos de teste do que se consegue escrever à mão, sem perder a confiança neles?  
**Ideia central:** dado sintético amplia a cobertura, mas só entra no golden depois de **filtro automático e revisão humana**. Gerar é barato, confiar é que custa.

Notebook **opcional** e independente dos principais. Cada seção abre com `preparar_secao` e roda sozinha. Onde o modelo é roteirizado, a célula diz.

Em produção, um modelo escreve as reclamações a partir de uma persona e de um cenário. Aqui as reclamações vêm de modelos de frase estilizados por persona, para o notebook ser reproduzível e gratuito; o restante do fluxo (dedup, filtro, revisão, variantes) é o mesmo.

In [1]:
from rangoja import config, dados, falhas, ferramentas, controle, observabilidade, relogio
from rangoja.armazem import Armazem
from rangoja.casos import caso
from rangoja.execucao import atender, continuar, aprovar, recusar, expirar_pendencias, reiniciar_runner, estado_persistido
from rangoja.notebook import (preparar_ambiente, preparar_secao, mostrar, verificar, verificar_falha,
                              resumo, amostra, matriz)
from rangoja.trajetoria import chamou, ferramenta_antes, contar_chamadas
from rangoja.versoes import v0, v1, v2, v3, montar
preparar_ambiente()

modo,replay
provedor,openai
modelo,gpt-4o-mini
chave de API,sim
observabilidade,ligado em http://127.0.0.1:6006


In [2]:
from rangoja import extras
from rangoja.modelo.roteiro import chamada, texto
from rangoja.notebook import Tabela

## X3.1 — Gerar por persona × cenário

7 cenários (um por categoria, mais fora do escopo) × 5 personas (formal, gíria, apressado, irritado, detalhista).

In [3]:
s = preparar_secao("X3.1")
itens = extras.gerar_reclamacoes()
print(len(itens), "reclamações geradas. Amostra:")
Tabela(["id", "persona", "cenário", "texto"], [[i.id, i.persona, i.cenario, i.texto] for i in itens[::9]])

105 reclamações geradas. Amostra:


id,persona,cenário,texto
S001,formal,item_faltando_bebida,"Prezados, faltou o refrigerante do pedido 4821. Solicito providências."
S010,irritado,item_faltando_bebida,MEU PEDIDO 4821 CHEGOU SEM A BEBIDA!!! ISSO É UM ABSURDO
S019,giria,item_faltando_sobremesa,"mano, A sobremesa do rango 4822 não veio!!"
S028,detalhista,item_faltando_sobremesa,A sobremesa do pedido 4822 não veio. Estou em casa com a família. Peço que verifiquem com atenção.
S037,apressado,atraso,o pedido 5310 demorou mais de uma hora para chegar
S046,formal,cobranca_dobro,"Prezados, o pedido 6002 foi cobrado em dobro no cartão. Solicito providências."
S055,irritado,cobranca_dobro,FUI COBRADO DUAS VEZES NO PEDIDO 6002!!! ISSO É UM ABSURDO
S064,giria,qualidade,"mano, A comida do rango 4821 veio fria!!"
S073,detalhista,qualidade,O pedido 4821 chegou com a embalagem aberta. Estou em casa com a família. Peço que verifiquem com atenção.
S082,apressado,golpe,recebi uma ligação dizendo ser do rangojá pedindo um pix foi golpe sobre o pedido 7730


## X3.2 — Filtrar duplicatas e inconsistências

Dois filtros automáticos. **Duplicatas**: similaridade de palavras ≥ 0,8. **Consistência**: um classificador de referência barato (palavras-chave, não o agente) tem de concordar com a categoria que o cenário promete, e o pedido citado tem de existir. Quem discorda **não é descartado**: vai para a fila de revisão humana, com o motivo.

In [4]:
unicos, duplicadas = extras.deduplicar(itens)
print(f"{len(itens)} geradas → {len(unicos)} únicas ({len(duplicadas)} quase-duplicatas removidas)")
aceitas, revisar = extras.filtrar(unicos)
print(f"{len(unicos)} únicas → {len(aceitas)} aceitas pelo filtro, {len(revisar)} para revisão humana")
Tabela(["id", "persona", "texto", "motivo"], [[i.id, i.persona, i.texto[:70], i.motivo] for i in revisar])

105 geradas → 60 únicas (45 quase-duplicatas removidas)
60 únicas → 51 aceitas pelo filtro, 9 para revisão humana


id,persona,texto,motivo
S061,formal,"Prezados, o pedido 4821 chegou sem tempero, praticamente cru. Solicito","referência classificou como item_faltando, esperado qualidade"
S062,formal,"Prezados, o pedido 4821 chegou com a embalagem aberta. Solicito provid","referência classificou como fora_do_escopo, esperado qualidade"
S066,giria,"mano, O rango 4821 veio sem tempero, praticamente cru!!","referência classificou como item_faltando, esperado qualidade"
S067,apressado,o pedido 4821 chegou com a embalagem aberta,"referência classificou como fora_do_escopo, esperado qualidade"
S070,irritado,"O PEDIDO 4821 CHEGOU SEM TEMPERO, PRATICAMENTE CRU!!! ISSO É UM ABSURD","referência classificou como item_faltando, esperado qualidade"
S071,irritado,O PEDIDO 4821 CHEGOU COM A EMBALAGEM ABERTA!!! ISSO É UM ABSURDO,"referência classificou como fora_do_escopo, esperado qualidade"
S073,detalhista,O pedido 4821 chegou com a embalagem aberta. Estou em casa com a famíl,"referência classificou como fora_do_escopo, esperado qualidade"
S074,detalhista,"O pedido 4821 chegou sem tempero, praticamente cru. Tava chovendo aqui","referência classificou como item_faltando, esperado qualidade"
S075,detalhista,O pedido 4821 chegou com a embalagem aberta. Já é a terceira vez que p,"referência classificou como fora_do_escopo, esperado qualidade"


In [5]:
verificar(len(duplicadas) > 0, "o filtro encontrou quase-duplicatas")
verificar(all(i.motivo for i in revisar) and len(revisar) > 0, "tudo que o filtro barrou tem um motivo escrito para o revisor")
verificar(len({i.cenario for i in aceitas}) == len(extras.CENARIOS), "todos os cenários continuam representados depois do filtro")

## X3.3 — Revisão humana

A fila acima é o trabalho do revisor. Para cada item ele decide: **aceitar** (a categoria esperada está certa e o classificador de referência errou, como em "embalagem aberta" = qualidade), **corrigir** a categoria, ou **descartar**. Só o que passou por essa decisão entra no golden, com a origem registrada (`origem: sintetico`), como as variantes que já estão em `casos.yaml`.

In [6]:
import yaml
exemplo = aceitas[0]
print(yaml.safe_dump([{
    "id": f"SYN-{exemplo.id}", "origem": "sintetico", "cliente_id": "101", "pedido_id": exemplo.pedido_id,
    "atendimentos": [{"mensagens": [exemplo.texto]}], "esperado": {"categoria": exemplo.categoria_esperada},
}], allow_unicode=True, sort_keys=False))

- id: SYN-S001
  origem: sintetico
  cliente_id: '101'
  pedido_id: '4821'
  atendimentos:
  - mensagens:
    - Prezados, faltou o refrigerante do pedido 4821. Solicito providências.
  esperado:
    categoria: item_faltando



## X3.4 — Variantes metamórficas e preservação de sentido

De um caso existente, geramos variantes (gíria, erro de digitação, ordem, ruído). Cada uma é **aceita** só se o sentido se preserva: o número do pedido continua lá e a categoria de referência não muda. Duas transformações são **defeituosas de propósito** (inverter o sentido, truncar) para mostrar que a checagem as pega.

In [7]:
base = next(i for i in aceitas if i.cenario == "item_faltando_bebida" and i.persona == "formal")
print("original:", base.texto, "\n")
linhas, aceitas_v, defeituosas = [], [], []
for nome, f in extras.TRANSFORMACOES.items():
    v = f(base.texto)
    ok, motivo = extras.preserva_sentido(base, v)
    linhas.append([nome, v, "aceita" if ok else f"rejeitada: {motivo}"])
    (aceitas_v if ok else defeituosas).append(nome)
Tabela(["transformação", "variante", "resultado"], linhas)

original: Prezados, faltou o refrigerante do pedido 4821. Solicito providências. 



transformação,variante,resultado
giria,"mano, Prezados, faltou o refrigerante do rango 4821. Solicito providências.",aceita
digitacao,"Prezados, flatou o refrigerante do pedido 4821. Sloicito providências.",aceita
ordem,"Solicito providências. Prezados, faltou o refrigerante do pedido 4821.",aceita
ruido,"Prezados, faltou o refrigerante do pedido 4821. Solicito providências. Tava chovendo aqui.",aceita
inversao (defeituosa),"Prezados, sobrou o refrigerante do pedido 4821. Solicito providências.",rejeitada: categoria virou fora_do_escopo
truncamento (defeituosa),"Prezados, faltou",rejeitada: perdeu o número do pedido


In [8]:
verificar(set(aceitas_v) == {"giria", "digitacao", "ordem", "ruido"}, "as quatro transformações legítimas foram aceitas: preservam pedido e categoria")
verificar(set(defeituosas) == {"inversao (defeituosa)", "truncamento (defeituosa)"}, "as duas defeituosas foram rejeitadas pela checagem")

As variantes aceitas alimentam a relação de **invariância** do NB04: o agente deve decidir igual para a original e para elas. A checagem é só tão boa quanto o classificador de referência; por isso o filtro também tem revisão humana.